# 第 5 章 · Tool Use（Executing Code，ADK）

比 `google_search` 更极端的一步：模型**自己写 Python**，在**服务端**执行，再把结果告诉你。

关键差别在 API 形状上——它不走 `tools=[...]`，而是单独的 `code_executor=` 字段。
代码里能同时看到三样东西：`executable_code`（模型写的代码）、`code_execution_result`
（执行输出）、以及最终的 `text`。**下面把这三样都打出来**，这才是这一节值得看的地方。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】向上找含 common/ 的目录；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, describe_adk, short_error, retry_call
from google.adk.agents import LlmAgent
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.runners import InMemoryRunner
from google.genai import types

USER_ID = "user1234"

# ★ 本章新增 BuiltInCodeExecutor：注意它**不在 tools=[] 里**，是单独的字段。
#   给了它之后，模型多了一种"动作"：写一段代码丢给服务端跑，拿回 stdout。
code_agent = LlmAgent(
    name="calculator_agent",
    model=adk_model(),                   # 【适配】书里 gemini-2.0-flash 已下线
    code_executor=BuiltInCodeExecutor(),
    instruction="""You are a calculator agent.
   When given a mathematical expression, write and execute Python code to calculate the result.
   Return only the final numerical result as plain text, without markdown or code blocks.
   """,
    description="Executes Python code to perform calculations.",
)


def call_agent(query: str) -> None:
    """跑一次查询，把「模型写的代码 / 执行输出 / 最终答复」分开打出来。"""
    runner = InMemoryRunner(code_agent)
    session_id = str(uuid.uuid4())
    # 【适配】create_session 是协程 → create_session_sync
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id=USER_ID, session_id=session_id
    )

    print(f"\n--- Running Query: {query} ---")
    final = "No final text response captured."
    try:
        # 【适配】中转站偶发 ReadTimeout，瞬时错误交给 retry_call 重试
        for event in retry_call(lambda: list(runner.run(
            user_id=USER_ID,
            session_id=session_id,
            new_message=types.Content(role="user", parts=[types.Part(text=query)]),
        )), label="代码执行 "):
            if not event.content:
                continue
            for part in event.content.parts or []:
                if getattr(part, "executable_code", None):
                    print("  模型写的代码:\n```python\n" + (part.executable_code.code or "").strip() + "\n```")
                elif getattr(part, "code_execution_result", None):
                    r = part.code_execution_result
                    print(f"  执行结果: {getattr(r, 'outcome', '?')} -> {(r.output or '').strip()}")
                elif getattr(part, "text", None) and part.text.strip():
                    final = part.text.strip()
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {short_error(e)}")
        return
    print("==> Final Agent Response:", final)


# 书里两个用例：一个纯算术，一个需要真的算（10! 靠心算容易错，适合看它是否真执行）
call_agent("Calculate the value of (5 + 7) * 3")
call_agent("What is 10 factorial?")


--- Running Query: Calculate the value of (5 + 7) * 3 ---


  模型写的代码:
```python
result = (5 + 7) * 3
print(result)
```
  执行结果: Outcome.OUTCOME_OK -> 36
==> Final Agent Response: 36

--- Running Query: What is 10 factorial? ---


  模型写的代码:
```python
import math
print(math.factorial(10))
```
  执行结果: Outcome.OUTCOME_OK -> 3628800
==> Final Agent Response: 3628800
